# Project 3: Internet Speed Tests

Zachary Griffin 

Data 101 

Spring 2026

## Introduction

My dataset is the "Cable Office Speed Tests"  Montgomery County's Open Data portal, which contains user-inputted results of internet speeds tests done across the county between 2016 and 2021. My goal is to create a linear model to predict one of the results of the internet speed test, the download speed. 
The full data set consisted of over 4000 entries, and some columns were questions that very few (88 or les) users had answered, so I did not include those columns in my model. I dropped NA values, and created an area column to group zip codes, so there were only 8 groups instead of around 30. 



| Variable | Description | 
|----------|-------------|
| Date | Date  of test | 
| Time | Time of test |
| How Are You Testing? | Work, Home, or Business | 
| Zip | Zip code of test |
| Provider | internet service provider |
| How are you connected| Wired or wireless |
| Advertised Download Speed | Download speed advertised. Mbits/sec |
| Satisfaction Rating | from 1- 5, 1 being the least |
| Advertised Price Per Mbps | Megabit cost advertised in US Dollars |


## Supervised Learning and PCA

This was immensely frustrating, as my dataset was largely categorical and not binary. I eventually discovered how to combine separatenly encoding the categorical variables and scaling the numerical variables, which allowed my model to work and give me an R-squared value, but does not give me coeffiecents for the importance matrix. As a result, I am unsure how to choose which features to remove from my model. 

Similarily, ran into problems with the PCA. The scree plot as shown in class would not work, although I was I able to run the pipeline specifying a desired varience level, I don't know how to ask to return that number of components. I'm unsure what to do next there.

## Conclusions

My initial supervised learning model has a horrible R-squared value of 0.137, and I am unsure what do do next due as discussed above. 

This data set probably would have been better for cluster analysis, but I *knew* I still didn't understand a Silhouette Score and thought I at least understood linear regression, and realizad way too late that sklearn did not have the same functionality I was expecting. 

## References

https://data.montgomerycountymd.gov/Consumer-Housing/Cable-Office-Speed-Test/tqbg-9hqn/about_data

https://jakevdp.github.io/PythonDataScienceHandbook/03.11-working-with-time-series.html

https://pandas.pydata.org/docs/getting_started/intro_tutorials/09_timeseries.html

https://www.kaggle.com/discussions/getting-started/584683

## Analysis and Code: Supervised Learning

### Import Libraries and Data

In [195]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.decomposition import PCA

In [124]:
df = pd.read_csv("Cable_Office_Speed_Test.csv", parse_dates = ["Date"])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4078 entries, 0 to 4077
Data columns (total 23 columns):
 #   Column                                                      Non-Null Count  Dtype         
---  ------                                                      --------------  -----         
 0   Response                                                    4078 non-null   int64         
 1   Date                                                        4078 non-null   datetime64[ns]
 2   Time                                                        4078 non-null   object        
 3   How Are You Testing                                         4078 non-null   object        
 4   Zip                                                         4078 non-null   int64         
 5   Provider                                                    3731 non-null   object        
 6   How are you connected                                       3643 non-null   object        
 7   Price Per Month         

In [79]:
df.head()

,Response,Date,Time,How Are You Testing,Zip,Provider,How are you connected,Price Per Month,Advertised Download Speed,Satisfaction Rating,...,Actual Price Per Mbps,Ping,Additional Comments,What kind of service plan do you have?,Where are you accessing the internet,What are you using the internet for?,Indoor,Do you buy mobile data alone or bundle with talk and text?,How much is your monthly bill?,Is your data plan unlimited?
0,42,2016-09-09,07:42,Work,20850,NaN,NaN,NaN,NaN,1.0,...,NaN,6.0,NaN,NaN,NaN,NaN,indoor,NaN,NaN,NaN
1,47,2016-09-13,09:54,Work,20850,NaN,NaN,NaN,NaN,3.0,...,NaN,7.0,NaN,NaN,NaN,NaN,indoor,NaN,NaN,NaN
2,49,2016-09-15,15:42,Work,20850,NaN,NaN,NaN,NaN,3.0,...,NaN,20.0,NaN,NaN,NaN,NaN,indoor,NaN,NaN,NaN
3,128,2016-10-14,08:54,Work,20850,NaN,NaN,NaN,NaN,NaN,...,NaN,8.0,NaN,NaN,NaN,NaN,indoor,NaN,NaN,NaN
4,129,2016-10-14,09:41,Work,20850,NaN,NaN,NaN,NaN,NaN,...,NaN,7.0,NaN,NaN,NaN,NaN,indoor,NaN,NaN,NaN


In [179]:
# I spent a fair amount of time trying to understnad timestamps, only for that to not work for the StandardScalar,
#df['Time'] = (df['Time'].astype(str) + ':00')
#df['Time'] = pd.to_timedelta(df['Time'])
# splitting the time column into hours and minutes, multiplying hours by 60, and adding the two back together for a 'total minutes' since midnight column
df[['hour', 'min']] = df['Time'].str.split(':', expand = True)

df['hour']= pd.to_numeric(df['hour'])
df['min'] = pd.to_numeric(df['min'])

df['total_min'] = df['hour'] *60 +df['min']

df['date_numeric'] = df['Date'].astype('int64') # the standard scaler can't parse the time stamp

df['Advertised Download Speed'] = pd.to_numeric(df['Advertised Download Speed'].str.replace(',', '')) 
df['Download Speed'] = pd.to_numeric(df1['Download Speed'].str.replace(',', '')) # these also need to be numeric! kind of importatn!


In [143]:
df.head()

,Response,Date,Time,testing,Zip,Provider,connection,Price Per Month,Advertised Download Speed,Satisfaction Rating,...,Where are you accessing the internet,What are you using the internet for?,Indoor,Do you buy mobile data alone or bundle with talk and text?,How much is your monthly bill?,Is your data plan unlimited?,hour,min,total_min,date_numeric
0,42,2016-09-09,07:42,Work,20850,NaN,NaN,NaN,NaN,1.0,...,NaN,NaN,indoor,NaN,NaN,NaN,7,42,462,1473379200000000000
1,47,2016-09-13,09:54,Work,20850,NaN,NaN,NaN,NaN,3.0,...,NaN,NaN,indoor,NaN,NaN,NaN,9,54,594,1473724800000000000
2,49,2016-09-15,15:42,Work,20850,NaN,NaN,NaN,NaN,3.0,...,NaN,NaN,indoor,NaN,NaN,NaN,15,42,942,1473897600000000000
3,128,2016-10-14,08:54,Work,20850,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,indoor,NaN,NaN,NaN,8,54,534,1476403200000000000
4,129,2016-10-14,09:41,Work,20850,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,indoor,NaN,NaN,NaN,9,41,581,1476403200000000000


In [180]:
df.rename(columns = {'How Are You Testing' : 'testing', 'How are you connected': 'connection', 'Advertised Price Per Mbps' : 'Advertised Price'},
          inplace = True)

## Data Cleaning 

Since very few users (between 6 and 88) answered  "Additional Comments", "What kind of service plan do you have?", "Where are you accessing the internet" ,"What are you using the internet for?", "Do you buy mobile data alone or bundle with talk and text?" "How much is your monthly bill?," and "Is your data plan unlimited?", I am not using those columns in my model. 

As best I can tell, the first column, "Response" is like an ID number for the test, and also does not belong in the model. 

"*Download Speed*", "*Upload Speed*" and "*Ping*" are all measures of connection speed and, along with *"Actual Price per Mbps"* are the results from the Internet Speed tests. Thus, I can only choose one of them to be my response variable. As *Download Speed* is the most important measure for most users, that is what I'm using, and I am thus dropping the others. 


In [181]:
# The columns that ARE being used are: 
titles = ['testing', 'Zip', 'Provider', 'connection', 'Advertised Download Speed', 'Satisfaction Rating',
    'Download Speed', 'Advertised Price', 'Indoor', 'date_numeric','total_min' ]

In [182]:
# new df is thus: 
df1 = df[titles]

df1 = df1.dropna() # drop the rows with NA values

df1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1423 entries, 17 to 4077
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   testing                    1423 non-null   object 
 1   Zip                        1423 non-null   int64  
 2   Provider                   1423 non-null   object 
 3   connection                 1423 non-null   object 
 4   Advertised Download Speed  1423 non-null   float64
 5   Satisfaction Rating        1423 non-null   float64
 6   Download Speed             1423 non-null   float64
 7   Advertised Price           1423 non-null   object 
 8   Indoor                     1423 non-null   object 
 9   date_numeric               1423 non-null   int64  
 10  total_min                  1423 non-null   int64  
dtypes: float64(3), int64(3), object(5)
memory usage: 133.4+ KB


In [183]:
sorted(df1['Zip'].unique()) # see which zip codes are represented 

[np.int64(20705),
 np.int64(20814),
 np.int64(20815),
 np.int64(20816),
 np.int64(20817),
 np.int64(20818),
 np.int64(20832),
 np.int64(20833),
 np.int64(20837),
 np.int64(20838),
 np.int64(20841),
 np.int64(20842),
 np.int64(20850),
 np.int64(20851),
 np.int64(20852),
 np.int64(20853),
 np.int64(20854),
 np.int64(20855),
 np.int64(20860),
 np.int64(20866),
 np.int64(20871),
 np.int64(20872),
 np.int64(20874),
 np.int64(20876),
 np.int64(20877),
 np.int64(20878),
 np.int64(20879),
 np.int64(20880),
 np.int64(20882),
 np.int64(20886),
 np.int64(20895),
 np.int64(20896),
 np.int64(20901),
 np.int64(20902),
 np.int64(20903),
 np.int64(20904),
 np.int64(20905),
 np.int64(20906),
 np.int64(20910),
 np.int64(20912)]

In [189]:
# Create Area Names for the Zip Codes 

def define_area(Zip):
    if Zip == 20705:
        return "Beltsville"
    elif 20814 <= Zip <= 20818: 
        return "Bethesda/Cabin John"
    elif 20832 <= Zip <= 20833 or Zip == 20860:
        return "Olney/Brookeville/Sandy Spring"
    elif Zip == 20837:
        return "Poolsville"
    elif Zip == 20838:
        return "Barnesville"
    elif 20850 <= Zip <= 20855:
        return "Rockville"
    elif 20866 == Zip:
        return "Burtonsville"
    elif Zip == 20895 or 20896:
        return "Kensington/Garret Park"
    elif 20901 <= Zip <=  20910:
        return "Silver Spring"
    else:
        return "Takoma Park"

df1['area'] = df1['Zip'].apply(define_area)

df1['Zip'].astype('category') # change Zip to categorical 

df1.head()
        

,testing,Zip,Provider,connection,Advertised Download Speed,Satisfaction Rating,Download Speed,Advertised Price,Indoor,date_numeric,total_min,area
17,Home,20877,Verizon,Wired connection,60.0,5.0,58.0,1.0,indoor,1477353600000000000,959,Kensington/Garret Park
22,Home,20877,Verizon,Wired connection,60.0,5.0,44.0,1.0,indoor,1477353600000000000,956,Kensington/Garret Park
26,Home,20850,Verizon,Wired connection,75.0,4.0,58.0,1.0,indoor,1477353600000000000,614,Rockville
37,Home,20850,Comcast,Wired connection,75.0,4.0,80.0,1.0,indoor,1477353600000000000,645,Rockville
52,Home,20850,Comcast,Wired connection,50.0,3.0,26.0,1.0,indoor,1477440000000000000,842,Rockville


In [ ]:
# How many columns need to be encoded?

In [95]:
df1['Provider'].unique() #see internet providers

array(['Verizon', 'Comcast', 'RCN', 'Verizon Fios', 'Verizon DSL',
       'Other', 'Windstream DSL'], dtype=object)

In [149]:
df1['testing'].unique() 

array(['Home'], dtype=object)

In [150]:
df1['connection'].unique()

array(['Wired connection',
       'Wired with WiFi, multiple devices in household', 'Wired',
       'Wired with WiFi, single device'], dtype=object)

In [136]:
df1['Indoor'].unique()

array(['indoor'], dtype=object)

In [139]:
df1['area'].unique()

array(['Kensington/Garret Park', 'Rockville', 'Bethesda/Cabin John',
       'Olney/Brookeville/Sandy Spring', 'Barnesville', 'Burtonsville',
       'Poolsville', 'Beltsville'], dtype=object)

In [190]:
#Provider, connection, and area need to be encoded.  The other two can be dropped as there is only one type

#df1_encoded = pd.get_dummies(df1, columns =['Provider', 'connection', 'area'], drop_first = True)

# this is getting immensely frustratng, as I can handle multiple options of categorical data in linear regression quite easily in R. 


df1['Advertised Price'] = df1['Advertised Price'].replace({r'\$': ''}, regex = True).astype(float) # forgot to remove the currency symbol earlier


df1.info()


<class 'pandas.core.frame.DataFrame'>
Index: 1423 entries, 17 to 4077
Data columns (total 12 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   testing                    1423 non-null   object 
 1   Zip                        1423 non-null   int64  
 2   Provider                   1423 non-null   object 
 3   connection                 1423 non-null   object 
 4   Advertised Download Speed  1423 non-null   float64
 5   Satisfaction Rating        1423 non-null   float64
 6   Download Speed             1423 non-null   float64
 7   Advertised Price           1423 non-null   float64
 8   Indoor                     1423 non-null   object 
 9   date_numeric               1423 non-null   int64  
 10  total_min                  1423 non-null   int64  
 11  area                       1423 non-null   object 
dtypes: float64(4), int64(3), object(5)
memory usage: 144.5+ KB


#### Multiple Linear Regression

In [201]:
# Split the Data
X = df1.drop(['Download Speed', 'testing', 'Indoor', 'Zip'], axis = 1) # dropping Zip to use Area instead -- 8 groups instead of 31
y = df1['Download Speed']

# can't apply the standarized scalar to everything as I have a lot of categorical data with more than just two options; using a pipeline instead

preprocessor = ColumnTransformer(
    transformers = [
        ( 'num', StandardScaler(), ['Advertised Download Speed', 'Satisfaction Rating', 'Advertised Price', 'date_numeric', 'total_min']),
        ('cat', OneHotEncoder(drop = 'first'), ['Provider', 'connection', 'area'])
    ])

model_pipeline = Pipeline(steps =[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

#Split the data        
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 58)

mr_model = model_pipeline.fit(X_train, y_train)

# coef_ doesn't work for the encoded things, I think, and I need to move on to the rest of the project
#I'm incredibly frustrated as dealing with the categorical variables would bave been extremely simple in R

# importance = pd.DataFrame({
#     'Feature': titles, 
#     'Standardized_Weight': mr_model.coef_}).sort_values(by='Standardized_Weight', key=abs, ascending=False)

# print(importance)

print(f"Multiple Regression Test R2: {r2_score(y_test, mr_model.predict(X_test)):.3f}")

Multiple Regression Test R2: 0.137


This is a very bad model but I can't get the importance to work to find out what to take OUT

## PCA 

In [204]:
X = df1.drop(['Download Speed', 'testing', 'Indoor', 'Zip'], axis = 1) # dropping Zip to use Area instead -- 8 groups instead of 31
y = df1['Download Speed']

# can't apply the standarized scalar to everything as I have a lot of categorical data with more than just two options; using a pipeline instead

preprocessor = ColumnTransformer(
    transformers = [
        ( 'num', StandardScaler(), ['Advertised Download Speed', 'Satisfaction Rating', 'Advertised Price', 'date_numeric', 'total_min']),
        ('cat', OneHotEncoder(drop = 'first'), ['Provider', 'connection', 'area'])
    ])

pca_pipeline = Pipeline(steps =[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components =0.85))
]) # putting the variance I wanted in here DID work, and I'm not entirely sure why that worked either


score = pca_pipeline.fit_transform(X)


In [202]:
# Scree Plot: See how many PC needed -- this didn't work, and I'm not entirely sure why
pca = PCA()

# plt.figure(figsize=(10, 5))
# plt.bar(range(1, len(pca.explained_variance_ratio_)+1), pca.explained_variance_ratio_, alpha=0.5, label='Individual')
# plt.step(range(1, len(pca.explained_variance_ratio_)+1), np.cumsum(pca.explained_variance_ratio_), where='mid', label='Cumulative')
# plt.axhline(y = .85, color = 'red', linestyle = 'dashed', linewidth = 1.5, label = '85% Explained')
# plt.ylabel('Explained variance ratio'); plt.xlabel('Component'); plt.title('Scree Plot'); plt.legend(); plt.show()

AttributeError: 'PCA' object has no attribute 'explained_variance_ratio_'

<Figure size 1000x500 with 0 Axes>